
# The Sims 4 review analysis

This notebook walks through the review analysis utilities one step at a time:
loading processed reviews, preprocessing text, counting words, scoring sentiment and
plotting each result directly in the cell that computes it.

Change `GAME` and `CUTOFF` below to analyse another processed game or move the
before/after boundary. The default preset keeps the Sims-specific keywords and
replacements from the original notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from steam_reviews.analysis.dataset import (
    add_period,
    add_year,
    game_name,
    load_reviews,
    split_by_cutoff,
)
from steam_reviews.analysis.plotting import (
    plot_boxplot,
    plot_distribution_fits,
    plot_engagement_boxplot,
    plot_scatter,
    plot_sentiment_categories,
    plot_sentiment_distribution,
    plot_sentiment_vs_recommendation,
    plot_word_frequencies,
    plot_wordcloud,
    plot_yearly_sentiment,
)
from steam_reviews.analysis.presets import preset_for
from steam_reviews.analysis.sentiment import add_sentiment, sentiment_summary
from steam_reviews.analysis.statistics import (
    engagement_split,
    pearson_correlations,
    yearly_mean,
)
from steam_reviews.analysis.text import (
    DEFAULT_EXTRA_STOPWORDS,
    PreprocessingConfig,
    prepare_documents,
    word_frequencies,
)

In [ ]:
pd.set_option("display.max_colwidth", 120)

In [ ]:
PRESET = preset_for("SIMS")
GAME = PRESET.app_id
CUTOFF = PRESET.cutoff

In [ ]:
def show(figure):
    """Display a figure in the notebook and release it afterwards."""
    plt.show()
    plt.close(figure)


## Load processed reviews

`load_reviews` reads `data/processed/reviews.csv`, selects one application ID and sorts
the rows by date.

In [ ]:
reviews = load_reviews(GAME)

In [ ]:
print(f"Game: {game_name(reviews)} (app {GAME})")
print(f"Reviews: {len(reviews)}")
print(f"Date range: {reviews['date'].min().date()} to {reviews['date'].max().date()}")
print(f"Recommendation counts: {reviews['recommendation'].value_counts().to_dict()}")

In [ ]:
reviews[["date", "text", "hours_played", "recommendation", "helpful_votes"]].head(10)


## Preprocess text

The preset preserves domain words such as `sims`, `dlc` and `buildmode`, and normalizes
variants such as `pack`, `packs` and `expansion` to `dlc`. The filtered `document`
column keeps only tokens that occur at least twice.

In [ ]:
prepared = prepare_documents(reviews, PRESET.preprocessing)

In [ ]:
print(f"Documents: {len(prepared.reviews)}")
print(f"Vocabulary: {len(prepared.vocabulary)} terms")
print(f"Sample vocabulary: {sorted(prepared.vocabulary)[:20]}")

In [ ]:
prepared.reviews[["text", "processed_review", "document"]].head(10)


## Word frequencies

Word frequencies are computed for the whole corpus and for the two periods.
`top_n` controls how many words are returned.

In [ ]:
before_documents, after_documents = split_by_cutoff(prepared.reviews, CUTOFF)
before_frequencies = word_frequencies(before_documents["document"], top_n=20)
after_frequencies = word_frequencies(after_documents["document"], top_n=20)

In [ ]:
print(f"Documents before {CUTOFF}: {len(before_documents)}")
print(f"Documents on/after {CUTOFF}: {len(after_documents)}")

In [ ]:
before_frequencies

In [ ]:
show(
    plot_word_frequencies(
        before_frequencies,
        title=f"Top words before {CUTOFF}",
    )
)
show(
    plot_word_frequencies(
        after_frequencies,
        title=f"Top words on/after {CUTOFF}",
    )
)


## Word clouds

A word cloud is generated from the filtered documents, so frequent stopwords and
one-off tokens are already removed.

In [ ]:
overall_text = " ".join(prepared.reviews["document"])
show(plot_wordcloud(overall_text, title=f"Word cloud for {game_name(reviews)}"))

In [ ]:
show(
    plot_wordcloud(
        " ".join(before_documents["document"]),
        title=f"Word cloud before {CUTOFF}",
    )
)
show(
    plot_wordcloud(
        " ".join(after_documents["document"]),
        title=f"Word cloud on/after {CUTOFF}",
    )
)


## Sentiment scoring

VADER computes a compound score between -1 and 1. `add_sentiment` also assigns the
categories `Positive`, `Neutral` and `Negative`. `add_year` and `add_period` add the
columns used by the later plots.

In [ ]:
sentiment = add_period(add_year(add_sentiment(prepared.reviews)), CUTOFF)

In [ ]:
sentiment[["date", "sentiment_score", "sentiment_category", "year", "period"]].head(10)

In [ ]:
before_sentiment, after_sentiment = split_by_cutoff(sentiment, CUTOFF)

In [ ]:
summary = pd.DataFrame(
    {
        "overall": pd.Series(sentiment_summary(sentiment)),
        "before": pd.Series(sentiment_summary(before_sentiment)),
        "after": pd.Series(sentiment_summary(after_sentiment)),
    }
)
summary


## Sentiment distributions

In [ ]:
show(
    plot_sentiment_distribution(
        before_sentiment["sentiment_score"],
        after_sentiment["sentiment_score"],
        cutoff=CUTOFF,
    )
)
show(
    plot_sentiment_categories(
        before_sentiment,
        after_sentiment,
        cutoff=CUTOFF,
    )
)


## Sentiment over time

In [ ]:
yearly = yearly_mean(sentiment)
yearly

In [ ]:
show(plot_yearly_sentiment(yearly))


## Distribution fits

The fitted Normal, Laplace and rescaled Beta distributions show which shape best
describes the sentiment scores in each period.

In [ ]:
show(
    plot_distribution_fits(
        before_sentiment["sentiment_score"],
        title=f"Before {CUTOFF}",
    )
)
show(
    plot_distribution_fits(
        after_sentiment["sentiment_score"],
        title=f"On/after {CUTOFF}",
    )
)


## Sentiment and recommendation

Split reviews by the author's recommendation to see whether the text sentiment agrees
with the recommendation label.

In [ ]:
show(plot_sentiment_vs_recommendation(sentiment))
show(
    plot_boxplot(
        sentiment,
        category_column="recommendation",
        value_column="sentiment_score",
        title="Sentiment score by recommendation",
    )
)


## Engagement and helpful votes

`engagement_split` marks reviews above the median playtime. The box plot compares
sentiment across recommendation and engagement groups.

In [ ]:
engaged = engagement_split(sentiment)
engaged[["hours_played", "helpful_votes", "engaged"]].describe()

In [ ]:
show(plot_engagement_boxplot(engaged))
show(
    plot_scatter(
        sentiment,
        x_column="hours_played",
        title="Sentiment vs hours played",
    )
)
show(
    plot_scatter(
        sentiment,
        x_column="helpful_votes",
        title="Sentiment vs helpful votes",
    )
)


## Correlations

`pearson_correlations` compares sentiment with playtime and helpful votes before and
after the cutoff.

In [ ]:
correlations = pearson_correlations(sentiment, CUTOFF)
correlations


## Yearly word clouds

The same word-cloud helper can be applied to any subset, for example each year.

In [ ]:
for year, group in sentiment.groupby("year"):
    text = " ".join(group["document"])
    if text.strip():
        show(plot_wordcloud(text, title=f"Word cloud for {year}"))


## Reusing the preprocessing rules

`PreprocessingConfig` can be adjusted without changing the notebook flow. This example
keeps the default stopwords and adds two extra words to remove.

In [ ]:
custom_config = PreprocessingConfig(
    preserve_keywords=PRESET.preprocessing.preserve_keywords,
    replacements=PRESET.preprocessing.replacements,
    extra_stopwords=DEFAULT_EXTRA_STOPWORDS | {"like", "really"},
)
custom_prepared = prepare_documents(reviews, custom_config)
custom_prepared.reviews[["text", "processed_review"]].head(5)